In [ ]:
import pandas as pd
import os
import shutil
import json

raw_path = './raw/'
bronze_path = './bronze/'
gold_path = './gold/'


os.makedirs(bronze_path, exist_ok=True)
os.makedirs(gold_path, exist_ok=True)

arquivos = [
    'olympic_paris_medals',
    'olympic_paris_medalist',
    'olympic_historic_Medal_Tally',
    'olympic_historic_Event_Results'
]


for nome in arquivos:
    
    df = pd.read_csv(f"{raw_path}{nome}.csv")
    df.to_parquet(f"{bronze_path}{nome}.parquet", index=False)
    
    shutil.copy(f"{raw_path}{nome}.json", f"{bronze_path}{nome}.json")

print("✅ Arquivos convertidos para Parquet e salvos na pasta bronze com os metadados!")

✅ Arquivos convertidos para Parquet e salvos na pasta bronze com os metadados!


In [ ]:

df_paris = pd.read_parquet(f"{bronze_path}olympic_paris_medals.parquet")
df_historico = pd.read_parquet(f"{bronze_path}olympic_historic_Medal_Tally.parquet")

df_integrado = pd.merge(
    df_paris, 
    df_historico, 
    left_on='country_code', 
    right_on='country_noc', 
    how='inner'
)

df_integrado.to_parquet(f"{bronze_path}dataset_integrado_paris_historico.parquet", index=False)
print("✅ JOIN realizado e dataset integrado salvo na pasta bronze!")

✅ JOIN realizado e dataset integrado salvo na pasta bronze!


In [ ]:
df_paris = pd.read_parquet(f"{bronze_path}olympic_paris_medals.parquet")

quadro_medalhas = df_paris.groupby('country')['medal_type'].value_counts().unstack(fill_value=0)

if 'Gold Medal' in quadro_medalhas.columns:
    quadro_medalhas = quadro_medalhas.rename(columns={'Gold Medal': 'Ouro', 'Silver Medal': 'Prata', 'Bronze Medal': 'Bronze'})

quadro_medalhas = quadro_medalhas.sort_values(by=['Ouro', 'Prata', 'Bronze'], ascending=False).reset_index()

quadro_medalhas.to_parquet(f"{gold_path}quadro_medalhas_final.parquet", index=False)
print("✅ Quadro de Medalhas gerado e salvo na pasta gold!")

✅ Quadro de Medalhas gerado e salvo na pasta gold!
